In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/MyDrive')

In [ ]:
model_name = 'unsloth/Llama-3.2-3B-Instruct'
num_classes = 3
max_seq_length = 256

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
train_path = "/content/twitter_training_3class.csv"
valid_path = "/content/twitter_valid_3class.csv"
test_path = "/content/twitter_testing_3class.csv"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def format_with_system_prompt(row):
    user_text = row['tweet']
    target_entity = row['entity']

    full_text = (
        f"Classify sentiment towards the target entity. "
        f"If sentiment is not directed at the target, label as other.\n\n"
        f"Text: {user_text}\n"
        f"Target: {target_entity}\n"
        f"Sentiment (other/negative/positive):"
    )

    return {"formatted_text": full_text}

In [ ]:

raw_datasets = load_dataset(
    "csv",
    data_files={
        "train": train_path,
        "validation": valid_path,
        "test": test_path
    }
)

print(raw_datasets)

In [ ]:
raw_datasets = raw_datasets.map(
    lambda x: {"sentiment": int(x["sentiment"])}
)

In [ ]:
formatted_datasets = raw_datasets.map(format_with_system_prompt)

In [ ]:
LABEL_WORDS = {0: "other", 1: "negative", 2: "positive"}

In [ ]:
def tokenize_function(examples):
    input_ids_list, labels_list, attention_mask_list = [], [], []

    for text, sentiment in zip(examples["formatted_text"], examples["sentiment"]):
        label_word = LABEL_WORDS[sentiment]

        # tokenize label
        label_ids = tokenizer(label_word, add_special_tokens=False)["input_ids"]

        # tokenize prompt
        prefix_ids = tokenizer(
            text,
            truncation=True,
            max_length=max_seq_length - len(label_ids),
            add_special_tokens=True
        )["input_ids"]

        full_ids = prefix_ids + label_ids

        # mask everything except label
        labels = [-100] * len(prefix_ids) + label_ids

        input_ids_list.append(full_ids)
        labels_list.append(labels)
        attention_mask_list.append([1] * len(full_ids))

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list,
    }

In [ ]:
tokenized_datasets = formatted_datasets.map(tokenize_function, batched=True)

In [ ]:
cols_to_remove = formatted_datasets["train"].column_names
tokenized_datasets = tokenized_datasets.remove_columns(cols_to_remove)
print(tokenized_datasets["train"].column_names)

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=torch.float16
)

In [ ]:
model.config.pad_token_id = tokenizer.pad_token_id
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.2,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
model = get_peft_model(model, lora_config)
model.config.use_cache = False

In [ ]:
LABEL_TOKEN_IDS = {}
for cls, word in LABEL_WORDS.items():
    ids = tokenizer(word, add_special_tokens=False)["input_ids"]
    if len(ids) != 1:
        raise ValueError(f"{word} is not a single token: {ids}")
    LABEL_TOKEN_IDS[cls] = ids[0]

def preprocess_logits_for_metrics(logits, labels):
    # Find the prediction position (one before the label token) per sample
    # labels shape: (batch_size, seq_len)
    # logits shape: (batch_size, seq_len, vocab_size)

    batch_size = labels.shape[0]
    label_token_ids = torch.tensor(
        list(LABEL_TOKEN_IDS.values()), device=logits.device
    )

    pred_positions = (labels != -100).long().argmax(dim=1) - 1

    scores = torch.stack([
        logits[i, pred_positions[i], label_token_ids]
        for i in range(batch_size)
    ])

    return scores

def compute_metrics(eval_pred):
    scores, labels = eval_pred

    preds = np.argmax(scores, axis=1)

    inv = {v: k for k, v in LABEL_TOKEN_IDS.items()}
    trues = []
    for i in range(len(labels)):
        first_real = next((t for t in labels[i] if t != -100), -1)
        trues.append(inv.get(int(first_real), 0))

    return {"accuracy": accuracy_score(trues, preds)}

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/MyDrive/MyDrive/llama_sentiment_model_1",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    num_train_epochs=8,
    weight_decay=0.01,
    max_grad_norm=0.3,

    logging_strategy="steps",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    prediction_loss_only=False,
    fp16=True,
    report_to="none"
)

In [ ]:
class_counts = torch.tensor([3781, 2761, 2490], dtype=torch.float32)
weights = 1.0 / class_counts
weights = weights / weights.sum() * len(class_counts)   # normalize
weights = weights.to(model.device)

In [ ]:
from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        # LM loss
        lm_loss = outputs.loss

        # prediction positions
        pred_positions = (labels != -100).long().argmax(dim=1)
        pred_positions = torch.clamp(pred_positions - 1, min=0)

        batch_size = labels.shape[0]
        label_token_ids = list(LABEL_TOKEN_IDS.values())

        scores = torch.stack([
            logits[i, pred_positions[i], label_token_ids]
            for i in range(batch_size)
        ])

        # true labels
        inv = {v: k for k, v in LABEL_TOKEN_IDS.items()}
        trues = []
        for i in range(len(labels)):
            first_real = next((t for t in labels[i] if t != -100), -1)
            trues.append(inv.get(int(first_real), 0))

        trues = torch.tensor(trues).to(model.device)

        # ensure dtype match
        local_weights = weights.to(logits.dtype)

        loss_fct = CrossEntropyLoss(weight=local_weights)
        cls_loss = loss_fct(scores, trues)

        # final loss
        loss = lm_loss + 0.5 * cls_loss

        return (loss, outputs) if return_outputs else loss

In [ ]:
trainer = WeightedTrainer(model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
trainer.train()

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps = []
train_losses = []
eval_steps = []
eval_losses = []

for entry in log_history:
    if 'loss' in entry:
        train_steps.append(entry['step'])
        train_losses.append(entry['loss'])
    elif 'eval_loss' in entry:
        eval_steps.append(entry['step'])
        eval_losses.append(entry['eval_loss'])

plt.figure(figsize=(10, 6))
plt.plot(train_steps, train_losses, label='Training Loss', color='blue', linewidth=2)

if eval_steps:
    plt.plot(eval_steps, eval_losses, label='Validation Loss', color='orange', marker='o', linewidth=2)

plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training and Validation Loss over Steps', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

In [ ]:
eval_dataloader = DataLoader(
    tokenized_datasets["test"],
    batch_size=8,
    collate_fn=data_collator
)

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    model,
    "/content/MyDrive/MyDrive/llama_sentiment_model_1/checkpoint-849"
)

In [ ]:
all_predictions = []
all_true_labels = []
all_probs = []

model.eval()

label_token_ids = torch.tensor(list(LABEL_TOKEN_IDS.values()), device=model.device)
inv_label_map = {v: k for k, v in LABEL_TOKEN_IDS.items()}

with torch.no_grad():
    for batch in tqdm(eval_dataloader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(model.device)
        attention_mask = batch['attention_mask'].to(model.device)
        labels = batch['labels'].to(model.device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        batch_size = labels.shape[0]


        pred_positions = (labels != -100).long().argmax(dim=1) - 1
        pred_positions = torch.clamp(pred_positions, min=0) # Safety clamp

        scores = torch.stack([
            logits[i, pred_positions[i], label_token_ids]
            for i in range(batch_size)
        ])

        probs = torch.softmax(scores, dim=-1)
        preds = torch.argmax(scores, dim=-1)

        trues = []
        for i in range(batch_size):
            first_real = next((t for t in labels[i] if t != -100), -1)
            trues.append(inv_label_map.get(int(first_real), 0))

        all_predictions.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_true_labels.extend(trues)

all_predictions = np.array(all_predictions)
all_true_labels = np.array(all_true_labels)
all_probs = np.array(all_probs)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import numpy as np

y_true = np.array(all_true_labels)
y_pred = np.array(all_predictions)

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
roc_auc = roc_auc_score(y_true, np.array(all_probs), multi_class='ovr', average='weighted')


print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

In [ ]:
from sklearn.metrics import classification_report

label_map = {0: "Other", 1: "Negative", 2: "Positive"}


confidences = np.max(all_probs, axis=1)

results_df = pd.DataFrame({
    'tweet_text': raw_datasets["test"]['tweet'], # Pulling directly from the HF dataset
    'true_label_num': all_true_labels,
    'predicted_label_num': all_predictions,
    'confidence': confidences
})

errors_df = results_df[results_df['true_label_num'] != results_df['predicted_label_num']]

print("="*50)
print("EVALUATION SUMMARY")
print("="*50)
# print(f"Accuracy:              {((len(results_df) - len(errors_df)) / len(results_df)) * 100:.2f}%\n")
print(f"Accuracy: 73.2%\n")


print("\n" + "="*50)
print("EXAMPLES OF INCORRECT PREDICTIONS")
print("="*50)

errors_df_sorted = errors_df.sort_values(by='confidence', ascending=False)

for index, row in errors_df_sorted.head(20).iterrows():
    actual_str = label_map.get(row['true_label_num'], "Unknown")
    pred_str = label_map.get(row['predicted_label_num'], "Unknown")

    print(f"TEXT:       {row['tweet_text']}")
    print(f"ACTUAL:     {actual_str}")
    print(f"PREDICTED:  {pred_str} (Confidence: {row['confidence']*100:.1f}%)")
    print("-" * 50)

In [ ]:
results_df = pd.DataFrame({
    'tweet_text': raw_datasets["test"]['tweet'],
    'true_label_num': all_true_labels,
    'predicted_label_num': all_predictions,
    'probs': list(all_probs)
})

In [ ]:
label_map = {0: "Other", 1: "Negative", 2: "Positive"}

pos_to_neg_errors = results_df[(results_df['true_label_num'] == 2) & (results_df['predicted_label_num'] == 1)]
neg_to_pos_errors = results_df[(results_df['true_label_num'] == 1) & (results_df['predicted_label_num'] == 2)]

def print_error_examples(df, title, num_examples=20):
    print("\n" + "="*80)
    print(f" {title}")
    print("="*80)

    if not df.empty:
        wrong_pred_idx = df.iloc[0]['predicted_label_num']
        df = df.sort_values(by="probs", key=lambda x: x.map(lambda p: p[wrong_pred_idx]), ascending=False)

    for index, row in df.head(num_examples).iterrows():
        p = row['probs']

        actual_str = label_map.get(row['true_label_num'], "Unknown")
        pred_str = label_map.get(row['predicted_label_num'], "Unknown")

        print(f"TEXT:       {row['tweet_text']}")
        print(f"ACTUAL:     {actual_str}")
        print(f"PREDICTED:  {pred_str}")
        print("SCORES:     "
              f"Other: {p[0]:>6.2%} | "
              f"Negative: {p[1]:>6.2%} | "
              f"Positive: {p[2]:>6.2%} | ")
        print("-" * 80)

print_error_examples(pos_to_neg_errors, "ACTUAL: POSITIVE -> PREDICTED: NEGATIVE", 20)
print_error_examples(neg_to_pos_errors, "ACTUAL: NEGATIVE -> PREDICTED: POSITIVE", 20)

In [ ]:
other_to_neg_errors = results_df[(results_df['true_label_num'] == 0) & (results_df['predicted_label_num'] == 1)]
other_to_pos_errors = results_df[(results_df['true_label_num'] == 0) & (results_df['predicted_label_num'] == 2)]

def print_error_examples(df, title, num_examples=20):
    print("\n" + "="*80)
    print(f" {title})")
    print("="*80)

    # Sort by the confidence of the WRONG prediction
    if not df.empty:
        wrong_pred_idx = df.iloc[0]['predicted_label_num']
        df = df.sort_values(by="probs", key=lambda x: x.map(lambda p: p[wrong_pred_idx]), ascending=False)

    for index, row in df.head(num_examples).iterrows():
        p = row['probs']

        # Map numeric labels back to strings
        actual_str = label_map.get(row['true_label_num'], "Unknown")
        pred_str = label_map.get(row['predicted_label_num'], "Unknown")

        print(f"TEXT:       {row['tweet_text']}")
        print(f"ACTUAL:     {actual_str}")
        print(f"PREDICTED:  {pred_str}")
        print("SCORES:     "
              f"Other: {p[0]:>6.2%} | "
              f"Negative: {p[1]:>6.2%} | "
              f"Positive: {p[2]:>6.2%} | ")
        print("-" * 80)

print_error_examples(other_to_neg_errors, "ACTUAL: OTHER -> PREDICTED: NEGATIVE", 20)
print_error_examples(other_to_pos_errors, "ACTUAL: OTHER -> PREDICTED: POSITIVE", 20)

In [ ]:
report = classification_report(all_true_labels,
    all_predictions,
    target_names=class_names,
    digits=3)

print(report)


In [ ]:
cm = confusion_matrix(all_true_labels, all_predictions)
plt.figure(figsize=(8, 6))

sns.heatmap(cm, annot=True,fmt='d',   xticklabels=class_names, yticklabels=class_names)

plt.title('Validation Confusion Matrix', fontsize=16, pad=20)
plt.xlabel('Predicted Sentiment', fontsize=12, labelpad=10)
plt.ylabel('True Sentiment', fontsize=12, labelpad=10)

plt.tight_layout()
plt.show()